---

## 1. Imports & Setup

In [1]:
# Core data processing
import pandas as pd
import numpy as np

# File handling
from pathlib import Path
import json

# Datetime utilities
from datetime import datetime, timezone

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Environment setup complete")
print(f"📦 pandas: {pd.__version__}")
print(f"📦 numpy: {np.__version__}")

✅ Environment setup complete
📦 pandas: 2.3.3
📦 numpy: 2.3.5


---

## 2. Load Combined Dataset

Load the merged dataset from notebook 02 containing both real and synthetic events.

In [2]:
# Define path to combined dataset
COMBINED_PATH = Path('artifacts/combined/events.parquet')

if not COMBINED_PATH.exists():
    raise FileNotFoundError(
        f"❌ Combined dataset not found: {COMBINED_PATH}\n"
        f"Please run notebook 02_merge_real_and_synthetic.ipynb first to create this file."
    )

print(f"📂 Loading combined dataset from: {COMBINED_PATH}")

# Load dataset
df_events = pd.read_parquet(COMBINED_PATH)

print(f"\n✅ Loaded combined dataset")
print(f"📊 Shape: {df_events.shape}")
print(f"📏 Columns: {list(df_events.columns)}")
print(f"\n📈 Dataset Stats:")
print(f"   Total events: {len(df_events):,}")
print(f"   Unique users: {df_events['user_id'].nunique():,}")
print(f"   Unique products: {df_events['product_id'].nunique():,}")
print(f"   Unique sessions: {df_events['session_id'].nunique():,}")
print(f"\n📊 Event type distribution:")
print(df_events['event_type'].value_counts())
print(f"\n📊 Source distribution:")
print(df_events['source'].value_counts())

📂 Loading combined dataset from: artifacts\combined\events.parquet

✅ Loaded combined dataset
📊 Shape: (120, 8)
📏 Columns: ['event_id', 'event_type', 'user_id', 'session_id', 'product_id', 'ts', 'properties', 'source']

📈 Dataset Stats:
   Total events: 120
   Unique users: 89
   Unique products: 85
   Unique sessions: 89

📊 Event type distribution:
event_type
view           100
add_to_cart      8
click            7
purchase         5
Name: count, dtype: int64

📊 Source distribution:
source
retailrocket    80
synthetic       40
Name: count, dtype: int64


In [3]:
# Parse timestamp as datetime with timezone
print("🔄 Parsing timestamps...")
df_events['ts_datetime'] = pd.to_datetime(df_events['ts'], format='ISO8601', utc=True)

# Sort by timestamp (important for temporal features)
print("🔄 Sorting events by timestamp...")
df_events = df_events.sort_values('ts_datetime').reset_index(drop=True)

print(f"\n✅ Timestamp processing complete")
print(f"   Earliest event: {df_events['ts_datetime'].min()}")
print(f"   Latest event: {df_events['ts_datetime'].max()}")
print(f"   Time span: {(df_events['ts_datetime'].max() - df_events['ts_datetime'].min()).days} days")

🔄 Parsing timestamps...
🔄 Sorting events by timestamp...

✅ Timestamp processing complete
   Earliest event: 2015-05-04 05:40:49+00:00
   Latest event: 2025-11-20 08:07:41.762415+00:00
   Time span: 3853 days


---

## 3. Define Reference Time

**Critical for preventing data leakage:** All features must be computed relative to a reference time that represents the training cutoff.

In production, this would be the current timestamp when features are computed. For this notebook, we use the latest event timestamp as the reference.

**Why this matters:** If we compute features using future information (e.g., "user purchased this item tomorrow"), our model will achieve unrealistic performance in training but fail in production. By fixing a reference time, we ensure features only use past data.

In [4]:
# Define reference time (latest event timestamp)
REFERENCE_TIME = df_events['ts_datetime'].max()

print(f"⏰ Reference Time: {REFERENCE_TIME}")
print(f"\n📝 All features will be computed as of this reference time.")
print(f"📝 Recency features measure time elapsed from each event to {REFERENCE_TIME}.")
print(f"\n✅ This prevents data leakage—features do not use future information.")

⏰ Reference Time: 2025-11-20 08:07:41.762415+00:00

📝 All features will be computed as of this reference time.
📝 Recency features measure time elapsed from each event to 2025-11-20 08:07:41.762415+00:00.

✅ This prevents data leakage—features do not use future information.


---

## 4. User-Level Feature Engineering

Create one row per user with aggregate features capturing their behavior patterns.

**Features to create:**
- `total_events`: Total number of events generated by user
- `views_count`: Number of view events
- `add_to_cart_count`: Number of add-to-cart events
- `purchase_count`: Number of purchase events
- `unique_products_interacted`: Number of distinct products user has interacted with
- `unique_sessions`: Number of distinct sessions
- `last_event_ts`: Timestamp of user's most recent event
- `recency_days`: Days since last event (from reference time)

In [5]:
print("👤 Engineering user-level features\n" + "="*50 + "\n")

# Basic aggregations per user
user_features = df_events.groupby('user_id').agg(
    total_events=('event_id', 'count'),
    unique_products_interacted=('product_id', 'nunique'),
    unique_sessions=('session_id', 'nunique'),
    last_event_ts=('ts_datetime', 'max')
).reset_index()

print(f"✅ Basic user aggregations computed: {user_features.shape}")

# Event type counts per user
# Create pivot table: rows=user_id, columns=event_type, values=count
event_type_counts = df_events.groupby(['user_id', 'event_type']).size().unstack(fill_value=0)
event_type_counts = event_type_counts.reset_index()

# Rename columns to match requirements
column_mapping = {
    'view': 'views_count',
    'add_to_cart': 'add_to_cart_count',
    'purchase': 'purchase_count'
}

# Rename existing columns
event_type_counts = event_type_counts.rename(columns=column_mapping)

# Ensure all required event type columns exist (fill missing with 0)
for col in ['views_count', 'add_to_cart_count', 'purchase_count']:
    if col not in event_type_counts.columns:
        event_type_counts[col] = 0

# Also handle 'click' if it exists
if 'click' in event_type_counts.columns:
    event_type_counts = event_type_counts.drop(columns=['click'])

print(f"✅ Event type counts computed")

# Merge event type counts with user features
user_features = user_features.merge(event_type_counts, on='user_id', how='left')

# Fill any NaN values with 0
user_features['views_count'] = user_features['views_count'].fillna(0).astype(int)
user_features['add_to_cart_count'] = user_features['add_to_cart_count'].fillna(0).astype(int)
user_features['purchase_count'] = user_features['purchase_count'].fillna(0).astype(int)

print(f"✅ Event type counts merged with user features")

# Recency: days since last event (from reference time)
user_features['recency_days'] = (REFERENCE_TIME - user_features['last_event_ts']).dt.total_seconds() / 86400

print(f"✅ Recency feature computed")
print(f"\n📊 User Features Shape: {user_features.shape}")
print(f"📏 Features: {list(user_features.columns)}")
print(f"\n📈 Sample user features:")
print(user_features.head(10))

👤 Engineering user-level features

✅ Basic user aggregations computed: (89, 5)
✅ Event type counts computed
✅ Event type counts merged with user features
✅ Recency feature computed

📊 User Features Shape: (89, 9)
📏 Features: ['user_id', 'total_events', 'unique_products_interacted', 'unique_sessions', 'last_event_ts', 'add_to_cart_count', 'purchase_count', 'views_count', 'recency_days']

📈 Sample user features:
   user_id  total_events  unique_products_interacted  unique_sessions  \
0  1000514             1                           1                1   
1  1015139             1                           1                1   
2  1027719             1                           1                1   
3  1049477             1                           1                1   
4  1066758             1                           1                1   
5   107717             1                           1                1   
6  1083494             1                           1                1   
7 

---

## 5. Item-Level Feature Engineering

Create one row per product with aggregate features capturing popularity and quality signals.

**Features to create:**
- `total_views`: Number of view events for this product
- `total_add_to_cart`: Number of add-to-cart events
- `total_purchases`: Number of purchase events
- `popularity_score`: Log-scaled popularity metric
- `conversion_rate`: Purchase conversion rate (purchases / views)
- `last_interaction_ts`: Timestamp of most recent interaction
- `recency_days`: Days since last interaction (from reference time)

In [6]:
print("📦 Engineering item-level features\n" + "="*50 + "\n")

# Basic aggregations per product
item_features = df_events.groupby('product_id').agg(
    last_interaction_ts=('ts_datetime', 'max')
).reset_index()

print(f"✅ Basic item aggregations computed: {item_features.shape}")

# Event type counts per product
item_event_counts = df_events.groupby(['product_id', 'event_type']).size().unstack(fill_value=0)
item_event_counts = item_event_counts.reset_index()

# Rename columns to match requirements
item_column_mapping = {
    'view': 'total_views',
    'add_to_cart': 'total_add_to_cart',
    'purchase': 'total_purchases'
}

item_event_counts = item_event_counts.rename(columns=item_column_mapping)

# Ensure all required columns exist
for col in ['total_views', 'total_add_to_cart', 'total_purchases']:
    if col not in item_event_counts.columns:
        item_event_counts[col] = 0

# Drop 'click' column if it exists
if 'click' in item_event_counts.columns:
    item_event_counts = item_event_counts.drop(columns=['click'])

print(f"✅ Event type counts computed")

# Merge event counts with item features
item_features = item_features.merge(item_event_counts, on='product_id', how='left')

# Fill any NaN values with 0
item_features['total_views'] = item_features['total_views'].fillna(0).astype(int)
item_features['total_add_to_cart'] = item_features['total_add_to_cart'].fillna(0).astype(int)
item_features['total_purchases'] = item_features['total_purchases'].fillna(0).astype(int)

print(f"✅ Event type counts merged with item features")

# Popularity score: log-scaled total interactions
total_interactions = item_features['total_views'] + item_features['total_add_to_cart'] + item_features['total_purchases']
item_features['popularity_score'] = np.log1p(total_interactions)

print(f"✅ Popularity score computed")

# Conversion rate: purchases / views (avoid division by zero)
item_features['conversion_rate'] = item_features['total_purchases'] / item_features['total_views'].replace(0, np.nan)
item_features['conversion_rate'] = item_features['conversion_rate'].fillna(0)

print(f"✅ Conversion rate computed")

# Recency: days since last interaction (from reference time)
item_features['recency_days'] = (REFERENCE_TIME - item_features['last_interaction_ts']).dt.total_seconds() / 86400

print(f"✅ Recency feature computed")
print(f"\n📊 Item Features Shape: {item_features.shape}")
print(f"📏 Features: {list(item_features.columns)}")
print(f"\n📈 Sample item features:")
print(item_features.head(10))

📦 Engineering item-level features

✅ Basic item aggregations computed: (85, 2)
✅ Event type counts computed
✅ Event type counts merged with item features
✅ Popularity score computed
✅ Conversion rate computed
✅ Recency feature computed

📊 Item Features Shape: (85, 8)
📏 Features: ['product_id', 'last_interaction_ts', 'total_add_to_cart', 'total_purchases', 'total_views', 'popularity_score', 'conversion_rate', 'recency_days']

📈 Sample item features:
  product_id              last_interaction_ts  total_add_to_cart  \
0          1 2025-11-20 08:07:40.962189+00:00                  3   
1     105632        2015-05-10 05:53:35+00:00                  0   
2     118195        2015-07-07 17:23:14+00:00                  0   
3     126365        2015-08-01 02:53:06+00:00                  0   
4     134825        2015-07-29 00:03:54+00:00                  0   
5     151283        2015-09-14 02:57:43+00:00                  0   
6     153909        2015-08-27 22:59:43+00:00                  1   
7  

---

## 6. User-Item Interaction Feature Engineering

Create one row per (user_id, product_id) pair capturing interaction strength.

**Features to create:**
- `interaction_count`: Number of interactions between user and product
- `last_interaction_ts`: Timestamp of most recent interaction
- `recency_days`: Days since last interaction (from reference time)
- `has_purchased`: Binary flag (1 if user purchased product, 0 otherwise)

In [7]:
print("🔗 Engineering user-item interaction features\n" + "="*50 + "\n")

# Basic aggregations per user-item pair
interaction_features = df_events.groupby(['user_id', 'product_id']).agg(
    interaction_count=('event_id', 'count'),
    last_interaction_ts=('ts_datetime', 'max')
).reset_index()

print(f"✅ Basic interaction aggregations computed: {interaction_features.shape}")

# Has purchased flag: check if user has any purchase event for this product
purchase_flags = df_events[df_events['event_type'] == 'purchase'].groupby(['user_id', 'product_id']).size().reset_index(name='purchase_event_count')
purchase_flags['has_purchased'] = 1
purchase_flags = purchase_flags[['user_id', 'product_id', 'has_purchased']]

# Merge purchase flags
interaction_features = interaction_features.merge(purchase_flags, on=['user_id', 'product_id'], how='left')
interaction_features['has_purchased'] = interaction_features['has_purchased'].fillna(0).astype(int)

print(f"✅ Purchase flags computed")

# Recency: days since last interaction (from reference time)
interaction_features['recency_days'] = (REFERENCE_TIME - interaction_features['last_interaction_ts']).dt.total_seconds() / 86400

print(f"✅ Recency feature computed")
print(f"\n📊 Interaction Features Shape: {interaction_features.shape}")
print(f"📏 Features: {list(interaction_features.columns)}")
print(f"\n📈 Sample interaction features:")
print(interaction_features.head(10))

🔗 Engineering user-item interaction features

✅ Basic interaction aggregations computed: (103, 4)
✅ Purchase flags computed
✅ Recency feature computed

📊 Interaction Features Shape: (103, 6)
📏 Features: ['user_id', 'product_id', 'interaction_count', 'last_interaction_ts', 'has_purchased', 'recency_days']

📈 Sample interaction features:
   user_id product_id  interaction_count       last_interaction_ts  \
0  1000514     427472                  1 2015-08-13 23:02:53+00:00   
1  1015139      17137                  1 2015-06-18 21:00:59+00:00   
2  1027719     287449                  1 2015-09-09 04:50:28+00:00   
3  1049477      23683                  1 2015-08-13 23:17:44+00:00   
4  1066758     221329                  1 2015-05-10 17:50:37+00:00   
5   107717     270487                  1 2015-07-18 20:37:26+00:00   
6  1083494     365948                  1 2015-09-03 22:28:54+00:00   
7  1159686     450082                  1 2015-09-07 22:49:44+00:00   
8  1200854     218746           

---

## 7. Light Sanity Checks

Quick validation to ensure features are correctly computed.

In [8]:
print("🔍 Feature Sanity Checks\n" + "="*50 + "\n")

# Check 1: Feature table shapes
print("1️⃣ FEATURE TABLE SHAPES:")
print(f"   User features: {user_features.shape}")
print(f"   Item features: {item_features.shape}")
print(f"   Interaction features: {interaction_features.shape}")
print(f"\n   Expected counts:")
print(f"   Users: {df_events['user_id'].nunique():,}")
print(f"   Products: {df_events['product_id'].nunique():,}")
print(f"   User-Product pairs: {df_events.groupby(['user_id', 'product_id']).ngroups:,}")

# Check 2: Display head of each table
print("\n2️⃣ USER FEATURES (first 5 rows):")
print(user_features.head())

print("\n3️⃣ ITEM FEATURES (first 5 rows):")
print(item_features.head())

print("\n4️⃣ INTERACTION FEATURES (first 5 rows):")
print(interaction_features.head())

# Check 3: Ensure no NaNs in critical fields
print("\n5️⃣ NULL VALUE CHECK:")
print("\n   User features:")
user_nulls = user_features.isnull().sum()
if user_nulls.sum() == 0:
    print("   ✅ No null values")
else:
    print("   ⚠️  Null values found:")
    print(user_nulls[user_nulls > 0])

print("\n   Item features:")
item_nulls = item_features.isnull().sum()
if item_nulls.sum() == 0:
    print("   ✅ No null values")
else:
    print("   ⚠️  Null values found:")
    print(item_nulls[item_nulls > 0])

print("\n   Interaction features:")
interaction_nulls = interaction_features.isnull().sum()
if interaction_nulls.sum() == 0:
    print("   ✅ No null values")
else:
    print("   ⚠️  Null values found:")
    print(interaction_nulls[interaction_nulls > 0])

print("\n" + "="*50)
print("✅ Sanity checks complete!")

🔍 Feature Sanity Checks

1️⃣ FEATURE TABLE SHAPES:
   User features: (89, 9)
   Item features: (85, 8)
   Interaction features: (103, 6)

   Expected counts:
   Users: 89
   Products: 85
   User-Product pairs: 103

2️⃣ USER FEATURES (first 5 rows):
   user_id  total_events  unique_products_interacted  unique_sessions  \
0  1000514             1                           1                1   
1  1015139             1                           1                1   
2  1027719             1                           1                1   
3  1049477             1                           1                1   
4  1066758             1                           1                1   

              last_event_ts  add_to_cart_count  purchase_count  views_count  \
0 2015-08-13 23:02:53+00:00                  0               0            1   
1 2015-06-18 21:00:59+00:00                  0               0            1   
2 2015-09-09 04:50:28+00:00                  0               0            1

---

## 8. Save Feature Artifacts

Save feature tables and schema documentation for downstream use in model training.

In [9]:
# Define output directory
OUTPUT_DIR = Path('artifacts/features')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"💾 Saving feature artifacts to: {OUTPUT_DIR}\n")

# Save user features
user_features_path = OUTPUT_DIR / 'user_features.parquet'
user_features.to_parquet(user_features_path, index=False, compression='snappy')
print(f"✅ Saved user features: {user_features_path}")
print(f"   Rows: {len(user_features):,}, Columns: {len(user_features.columns)}")
print(f"   File size: {user_features_path.stat().st_size / 1024:.2f} KB")

# Save item features
item_features_path = OUTPUT_DIR / 'item_features.parquet'
item_features.to_parquet(item_features_path, index=False, compression='snappy')
print(f"\n✅ Saved item features: {item_features_path}")
print(f"   Rows: {len(item_features):,}, Columns: {len(item_features.columns)}")
print(f"   File size: {item_features_path.stat().st_size / 1024:.2f} KB")

# Save interaction features
interaction_features_path = OUTPUT_DIR / 'interaction_features.parquet'
interaction_features.to_parquet(interaction_features_path, index=False, compression='snappy')
print(f"\n✅ Saved interaction features: {interaction_features_path}")
print(f"   Rows: {len(interaction_features):,}, Columns: {len(interaction_features.columns)}")
print(f"   File size: {interaction_features_path.stat().st_size / 1024:.2f} KB")

💾 Saving feature artifacts to: artifacts\features

✅ Saved user features: artifacts\features\user_features.parquet
   Rows: 89, Columns: 9
   File size: 8.58 KB

✅ Saved item features: artifacts\features\item_features.parquet
   Rows: 85, Columns: 8
   File size: 7.83 KB

✅ Saved interaction features: artifacts\features\interaction_features.parquet
   Rows: 103, Columns: 6
   File size: 7.52 KB


In [10]:
# Create feature schema documentation
feature_schema = {
    "reference_time": str(REFERENCE_TIME),
    "description": "Feature tables for recommender system training",
    "user_features": {
        "description": "Aggregate features per user capturing behavior patterns",
        "key": "user_id",
        "row_count": len(user_features),
        "features": {
            "user_id": "Unique user identifier",
            "total_events": "Total number of events generated by user",
            "unique_products_interacted": "Number of unique products user has interacted with",
            "unique_sessions": "Number of unique sessions for user",
            "last_event_ts": "Timestamp of user's most recent event",
            "views_count": "Number of view events",
            "add_to_cart_count": "Number of add-to-cart events",
            "purchase_count": "Number of purchase events",
            "recency_days": "Days since last event (measured from reference time)"
        }
    },
    "item_features": {
        "description": "Aggregate features per product capturing popularity and quality",
        "key": "product_id",
        "row_count": len(item_features),
        "features": {
            "product_id": "Unique product identifier",
            "last_interaction_ts": "Timestamp of most recent interaction with product",
            "total_views": "Number of view events for this product",
            "total_add_to_cart": "Number of add-to-cart events for this product",
            "total_purchases": "Number of purchase events for this product",
            "popularity_score": "Log-scaled popularity metric (log1p of total interactions)",
            "conversion_rate": "Purchase conversion rate (total_purchases / total_views)",
            "recency_days": "Days since last interaction (measured from reference time)"
        }
    },
    "interaction_features": {
        "description": "Features for each user-item pair capturing interaction strength",
        "key": ["user_id", "product_id"],
        "row_count": len(interaction_features),
        "features": {
            "user_id": "User identifier",
            "product_id": "Product identifier",
            "interaction_count": "Number of interactions between user and product",
            "last_interaction_ts": "Timestamp of most recent interaction between user and product",
            "has_purchased": "Binary flag: 1 if user has purchased product, 0 otherwise",
            "recency_days": "Days since last interaction (measured from reference time)"
        }
    }
}

# Save schema as JSON
schema_path = OUTPUT_DIR / 'feature_schema.json'
with open(schema_path, 'w') as f:
    json.dump(feature_schema, f, indent=2)

print(f"\n✅ Saved feature schema: {schema_path}")
print(f"   File size: {schema_path.stat().st_size / 1024:.2f} KB")
print(f"\n📝 Schema includes:")
print(f"   - Reference time: {REFERENCE_TIME}")
print(f"   - User features: {len(user_features)} rows, {len(user_features.columns)} columns")
print(f"   - Item features: {len(item_features)} rows, {len(item_features.columns)} columns")
print(f"   - Interaction features: {len(interaction_features)} rows, {len(interaction_features.columns)} columns")


✅ Saved feature schema: artifacts\features\feature_schema.json
   File size: 2.27 KB

📝 Schema includes:
   - Reference time: 2025-11-20 08:07:41.762415+00:00
   - User features: 89 rows, 9 columns
   - Item features: 85 rows, 8 columns
   - Interaction features: 103 rows, 6 columns
